### Load Azure Fabric quotas

Loads the raw JSON files written by the **Load Quota** pipeline (`Files/Data/Quotas`) into the `quota_fabric` table.

The pipeline calls the Azure Resource Manager batch endpoint with one `Microsoft.Fabric/locations/{region}/usages` request (api-version `2023-11-01`) per region and subscription. Each item of a response is one quota with its identifier (`name.value`), its display name (`name.localizedValue`), its `unit`, the current usage (`currentValue`) and the configured `limit`.

- One snapshot per day: running the notebook again the same day replaces the data of the subscriptions collected by that run (data of other subscriptions and of other days is untouched).
- Requests that failed (missing permission, subscription not found, region not available...) are reported in the output and skipped. They never prevent the load of the other subscriptions and regions.

In [ ]:
from datetime import datetime
import json
from notebookutils import mssparkutils
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, ArrayType


In [ ]:
# Schema of the Azure Resource Manager batch response written by the Load Quota pipeline
schemaJson = StructType([
    StructField("responses", ArrayType(
        StructType([
            StructField("name", StringType(), True),  # name of the batch request = Azure region
            StructField("httpStatusCode", IntegerType(), True),
            StructField("content", StructType([
                StructField("error", StructType([
                    StructField("code", StringType(), True),
                    StructField("message", StringType(), True)
                ]), True),
                StructField("nextLink", StringType(), True),
                StructField("value", ArrayType(
                    StructType([
                        StructField("name", StructType([
                            StructField("value", StringType(), True),
                            StructField("localizedValue", StringType(), True)
                        ]), True),
                        StructField("unit", StringType(), True),
                        StructField("currentValue", LongType(), True),
                        StructField("limit", LongType(), True),
                        StructField("id", StringType(), True)
                    ])
                ), True)
            ]), True)
        ])
    ), True)
])

In [ ]:
pathQuotas = "Files/Data/Quotas"
tableName = "quota_fabric"

quotasListFiles = []
if mssparkutils.fs.exists(pathQuotas):
    quotasListFiles = [f for f in mssparkutils.fs.ls(pathQuotas) if f.isFile and f.name.endswith(".json")]

# We only keep 1 snapshot per day
now = datetime.now()
current_date = int(now.strftime("%Y%m%d"))

print(f"{len(quotasListFiles)} quota file(s) found in {pathQuotas}")

In [ ]:
# One row per batch request (subscription + region) with its HTTP status and either the quotas or the error
df_responses = None
total_requests = 0

if quotasListFiles:
    df_responses = (
        spark.read.option("multiline", "true").schema(schemaJson).json([f.path for f in quotasListFiles])
        .withColumn("SourceFile", F.element_at(F.split(F.input_file_name(), "/"), -1))
        .select("SourceFile", F.explode("responses").alias("r"))
        .select(
            F.split("SourceFile", "_").getItem(1).cast("int").alias("SubscriptionKey"),  # file name: <SubscriptionId>_<SubscriptionKey>_regionsN.json
            F.col("r.name").alias("RegionId"),
            F.col("r.httpStatusCode").alias("HttpStatus"),
            F.col("r.content.error.code").alias("ErrorCode"),
            F.col("r.content.error.message").alias("ErrorMessage"),
            F.col("r.content.nextLink").alias("NextLink"),
            F.col("r.content.value").alias("Quotas")
        )
        .cache()
    )
    total_requests = df_responses.count()
    print(f"{total_requests} region request(s) read")

In [ ]:
# Report the requests that did not return quotas. They are skipped, the other ones are loaded.
failed_requests = 0

if df_responses is not None:
    failed_groups = (
        df_responses.where(F.col("HttpStatus").isNull() | (F.col("HttpStatus") != 200))
        .groupBy("HttpStatus", "ErrorCode")
        .agg(
            F.count("*").alias("Requests"),
            F.countDistinct("SubscriptionKey").alias("Subscriptions"),
            F.first("RegionId").alias("ExampleRegion"),
            F.substring(F.first("ErrorMessage"), 1, 300).alias("ExampleMessage")
        )
        .orderBy("HttpStatus", "ErrorCode")
        .collect()
    )
    for g in failed_groups:
        failed_requests += g["Requests"]
        print(f"WARNING - {g['Requests']} of {total_requests} request(s) skipped: HTTP {g['HttpStatus']} {g['ErrorCode']} on {g['Subscriptions']} subscription(s). Example (region {g['ExampleRegion']}): {g['ExampleMessage']}")
        if g["HttpStatus"] in (401, 403):
            print("  -> The identity of the 'fca azure management' connection must be able to read the Fabric quotas of these subscriptions (Contributor role, see Quota.md).")

    paged_requests = df_responses.where(F.col("NextLink").isNotNull()).count()
    if paged_requests > 0:
        print(f"WARNING - {paged_requests} response(s) contain a nextLink, only the first page of quotas is loaded.")

    if failed_requests == 0:
        print("All requests succeeded")

In [ ]:
# One row per subscription, region and quota. Usage and Limit are the values returned by the API (currentValue and limit)
loaded_rows = 0

if df_responses is not None:
    df_quota = (
        df_responses.where((F.col("HttpStatus") == 200) & F.col("SubscriptionKey").isNotNull())
        .select("SubscriptionKey", "RegionId", F.explode("Quotas").alias("q"))
        .select(
            F.col("RegionId"),
            F.col("q.currentValue").cast("int").alias("Usage"),
            F.col("q.limit").cast("int").alias("Limit"),
            F.lit(current_date).alias("DateKey"),
            F.col("SubscriptionKey"),
            F.col("q.name.value").alias("QuotaId"),
            F.col("q.name.localizedValue").alias("QuotaName"),
            F.col("q.unit").alias("Unit"),
            F.lit(now).alias("CollectionTimestamp")
        )
        .where(F.col("QuotaId").isNotNull())
        .dropDuplicates(["SubscriptionKey", "RegionId", "QuotaId"])
        .cache()
    )
    loaded_rows = df_quota.count()

In [ ]:
# Replace the snapshot of the day for the subscriptions collected by this run. If nothing could be collected, the existing data is kept.
loaded_subscriptions = 0

if loaded_rows > 0:
    subscription_keys = sorted(r["SubscriptionKey"] for r in df_quota.select("SubscriptionKey").distinct().collect())
    loaded_subscriptions = len(subscription_keys)
    keys = ",".join(str(k) for k in subscription_keys)

    (df_quota.write.format("delta").mode("overwrite")
        .option("replaceWhere", f"DateKey = {current_date} AND SubscriptionKey IN ({keys})")
        .option("mergeSchema", "true")  # adds the quota columns if the table comes from an older FCA version
        .saveAsTable(tableName))
    print(f"Load Done: {loaded_rows} quota row(s) for {loaded_subscriptions} subscription(s) in {tableName} (DateKey {current_date})")
else:
    print("No quota data to load, the existing data of the table is left untouched")

In [ ]:
# Clean raw files
for quotaFile in quotasListFiles:
    mssparkutils.fs.rm(quotaFile.path)

print("Clean Done")

mssparkutils.notebook.exit(json.dumps({
    "dateKey": current_date,
    "requests": total_requests,
    "failedRequests": failed_requests,
    "loadedRows": loaded_rows,
    "loadedSubscriptions": loaded_subscriptions
}))